<a href="https://colab.research.google.com/github/priyanka260604/first_gitrepo/blob/main/CNN_fashion_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [43]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [44]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [45]:
df=pd.read_csv('/content/fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [46]:
df.shape

(60000, 785)

In [47]:
# train test split

X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [48]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [49]:
# scaling the feautures
X_train = X_train/255.0
X_test = X_test/255.0

In [60]:
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features=torch.tensor(features,dtype=torch.float32).reshape(-1,1,28,28)
    self.labels=torch.tensor(labels,dtype=torch.long)

  def __len__(self):
    return len(self.features)
  def __getitem__(self, index):
    return self.features[index],self.labels[index]

In [61]:
train_dataset=CustomDataset(X_train,y_train)
test_dataset=CustomDataset(X_test,y_test)

In [62]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

In [63]:
class MyNN(nn.Module):
  def __init__(self,input_features):
    super().__init__()

    self.features=nn.Sequential(
        nn.Conv2d(input_features,32,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),
        nn.BatchNorm2d(32),
        nn.Conv2d(32,64,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )
    self.classifier=nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*7*7,128),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(64,10)
    )

  def forward(self,x):
    x=self.features(x)
    x=self.classifier(x)

    return x

In [64]:
learning_rate=0.01
epochs=100

In [65]:
model=MyNN(1)

model.to('cuda')
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=learning_rate,weight_decay=1e-4)

In [66]:
#training loop

for epoch in range(epochs):
  total_epoch_loss=0

for batch_features,batch_labels in train_loader:
  batch_features,batch_labels=batch_features.to('cuda'),batch_labels.to('cuda')
  outputs=model(batch_features)
  loss=criterion(outputs,batch_labels)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  total_epoch_loss+=loss.item()

  avg_loss=total_epoch_loss/len(train_loader)
  print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')

Epoch [100/100], Loss: 0.0016
Epoch [100/100], Loss: 0.0043
Epoch [100/100], Loss: 0.0063
Epoch [100/100], Loss: 0.0080
Epoch [100/100], Loss: 0.0094
Epoch [100/100], Loss: 0.0109
Epoch [100/100], Loss: 0.0125
Epoch [100/100], Loss: 0.0142
Epoch [100/100], Loss: 0.0161
Epoch [100/100], Loss: 0.0172
Epoch [100/100], Loss: 0.0181
Epoch [100/100], Loss: 0.0195
Epoch [100/100], Loss: 0.0210
Epoch [100/100], Loss: 0.0222
Epoch [100/100], Loss: 0.0234
Epoch [100/100], Loss: 0.0243
Epoch [100/100], Loss: 0.0253
Epoch [100/100], Loss: 0.0264
Epoch [100/100], Loss: 0.0275
Epoch [100/100], Loss: 0.0293
Epoch [100/100], Loss: 0.0303
Epoch [100/100], Loss: 0.0313
Epoch [100/100], Loss: 0.0327
Epoch [100/100], Loss: 0.0338
Epoch [100/100], Loss: 0.0349
Epoch [100/100], Loss: 0.0360
Epoch [100/100], Loss: 0.0372
Epoch [100/100], Loss: 0.0380
Epoch [100/100], Loss: 0.0391
Epoch [100/100], Loss: 0.0399
Epoch [100/100], Loss: 0.0407
Epoch [100/100], Loss: 0.0415
Epoch [100/100], Loss: 0.0423
Epoch [100

In [67]:
model.eval()

MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [68]:
from numpy import corrcoef
#Evaluation on test data
total=0
correct=0

with torch.no_grad():
  for batch_features,batch_labels in test_loader:
    batch_features,batch_labels=batch_features.to('cuda'),batch_labels.to('cuda')
    outputs=model(batch_features)
    _,predicted=torch.max(outputs,1)
    total+=batch_labels.size(0)
    correct+=(predicted==batch_labels).sum().item()

accuracy=correct/total
print(f'Accuracy on test data: {accuracy:}%')

Accuracy on test data: 0.82375%


In [59]:
from numpy import corrcoef
#Evaluation on train data
total=0
correct=0

with torch.no_grad():
  for batch_features,batch_labels in train_loader:
    batch_features,batch_labels=batch_features.to('cuda'),batch_labels.to('cuda')
    outputs=model(batch_features)
    _,predicted=torch.max(outputs,1)
    total+=batch_labels.size(0)
    correct+=(predicted==batch_labels).sum().item()

accuracy=correct/total
print(f'Accuracy on train data: {accuracy:}%')

Accuracy on train data: 0.8242291666666667%
